# ConvNeXt backbone check (v2c)

Trains 3 heads × 3 seeds with a ConvNeXt-Tiny backbone and saves predictions to Google Drive (`MyDrive/emotion_v2c`).

**Before running:** Runtime → Change runtime type → **A100** (or L4). Then Runtime → **Run all**.

If Colab disconnects, just **Run all** again: finished models are skipped and it continues where it stopped.

In [ ]:
# 1. GPU check
!nvidia-smi -L

In [ ]:
# 2. Code + dependencies (do NOT install requirements.txt: it breaks Colab's numpy)
import os
if not os.path.exists('/content/Emotion-Modelling'):
    !git clone -q https://github.com/Hassan-Naeem-code/Emotion-Modelling.git /content/Emotion-Modelling
else:
    !git -C /content/Emotion-Modelling pull -q
%cd /content/Emotion-Modelling
!pip -q install timm pyyaml

In [ ]:
# 3. Dataset from Drive to the fast local disk (~2 min)
from google.colab import drive
drive.mount('/content/drive')
if not os.path.exists('data/raw/AffectNetVA/Train/images'):
    !mkdir -p data/raw && unzip -q -o /content/drive/MyDrive/AffectNetVA.zip -d data/raw/
!ls data/raw/AffectNetVA/Train/images | wc -l   # expect 58803

In [ ]:
# 4. Train + predict (long: ~25 min per model on A100, 9 models). Resumable.
!mkdir -p /content/drive/MyDrive/emotion_v2c
!PREFIX=v2c OUT=/content/drive/MyDrive/emotion_v2c SEEDS="1 2 3" METHODS="evidential gaussian_nll baseline" bash scripts/run_v2.sh

In [ ]:
# 5. Progress: should end with 12 files (9 models + 3 MC-dropout)
!ls -la /content/drive/MyDrive/emotion_v2c/preds/